In [ ]:
# Tham số có thể điều chỉnh khi chạy với Papermill
RULES_PATH = "data/processed/rules_apriori_filtered.csv"
OUTPUT_DIR = "data/processed"

# Ngưỡng phân tích
LOW_SUPPORT_THRESHOLD = 0.02  # Support dưới 2% được coi là thấp
HIGH_LIFT_THRESHOLD = 10.0    # Lift trên 10 được coi là cao
TOP_N_RULES = 20              # Số luật top để phân tích

# Biểu đồ
PLOT_SUPPORT_LIFT_SCATTER = True
PLOT_TOP_NICHE_RULES = True
PLOT_SEGMENT_ANALYSIS = True

In [ ]:
# Import thư viện cần thiết
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

# Cấu hình hiển thị
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✓ Đã import thành công các thư viện")

In [ ]:
# Load dữ liệu luật kết hợp
rules_df = pd.read_csv(RULES_PATH)

print(f"Tổng số luật: {len(rules_df)}")
print(f"\nCác cột trong dữ liệu: {list(rules_df.columns)}")
print(f"\nThống kê mô tả:")
print(rules_df[['support', 'confidence', 'lift', 'leverage', 'conviction']].describe())

---
## PHẦN 1: XEM XÉT CÁC LUẬT CÓ SUPPORT THẤP NHƯNG LIFT CAO

Các luật này đại diện cho **các phân khúc thị trường nhỏ (niche market)** với mối liên kết mạnh giữa các sản phẩm.

In [ ]:
# Lọc các luật có support thấp và lift cao
niche_rules = rules_df[
    (rules_df['support'] < LOW_SUPPORT_THRESHOLD) & 
    (rules_df['lift'] > HIGH_LIFT_THRESHOLD)
].copy()

niche_rules = niche_rules.sort_values('lift', ascending=False)

print(f"Tìm thấy {len(niche_rules)} luật có support < {LOW_SUPPORT_THRESHOLD} và lift > {HIGH_LIFT_THRESHOLD}")
print(f"\n{'='*80}")
print(f"TOP {min(TOP_N_RULES, len(niche_rules))} LUẬT NICHE (Support thấp, Lift cao):")
print(f"{'='*80}\n")

# Hiển thị top luật
display_cols = ['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift', 'leverage']
top_niche = niche_rules.head(TOP_N_RULES)

for idx, row in top_niche.iterrows():
    print(f"Luật #{idx+1}:")
    print(f"  {row['antecedents_str']} → {row['consequents_str']}")
    print(f"  Support: {row['support']:.4f} ({row['support']*100:.2f}%)")
    print(f"  Confidence: {row['confidence']:.4f} ({row['confidence']*100:.2f}%)")
    print(f"  Lift: {row['lift']:.2f}")
    print(f"  Leverage: {row['leverage']:.6f}")
    print()

print(f"\nThống kê các luật niche:")
print(niche_rules[['support', 'confidence', 'lift']].describe())

In [ ]:
if PLOT_SUPPORT_LIFT_SCATTER:
    # Biểu đồ scatter: Support vs Lift
    fig = px.scatter(
        rules_df,
        x='support',
        y='lift',
        size='confidence',
        color='confidence',
        hover_data=['antecedents_str', 'consequents_str', 'confidence', 'lift'],
        title='Phân Bố Luật: Support vs Lift (Size = Confidence)',
        labels={'support': 'Support', 'lift': 'Lift', 'confidence': 'Confidence'},
        color_continuous_scale='viridis',
        width=1000,
        height=600
    )
    
    # Thêm vùng highlight cho niche rules
    fig.add_hrect(
        y0=HIGH_LIFT_THRESHOLD, y1=rules_df['lift'].max(),
        fillcolor="red", opacity=0.1,
        annotation_text="Lift cao", annotation_position="top left"
    )
    
    fig.add_vrect(
        x0=0, x1=LOW_SUPPORT_THRESHOLD,
        fillcolor="blue", opacity=0.1,
        annotation_text="Support thấp", annotation_position="bottom right"
    )
    
    fig.update_layout(
        xaxis_title="Support (Độ phổ biến)",
        yaxis_title="Lift (Độ mạnh liên kết)"
    )
    
    fig.show()
    
    # Export ảnh cho README
    import os
    os.makedirs('images', exist_ok=True)
    import plotly.io as pio
    try:
        pio.write_image(fig, 'images/support_lift_tradeoff.png', width=1200, height=800)
        print("✅ Đã lưu ảnh: images/support_lift_tradeoff.png")
    except Exception as e:
        print(f"⚠️ Không thể export Plotly image: {e}")
        print("   Cài đặt kaleido: pip install kaleido")
    
    print("✓ Vùng giao giữa 'Support thấp' và 'Lift cao' là các luật niche đáng chú ý")

In [ ]:
if PLOT_TOP_NICHE_RULES and len(niche_rules) > 0:
    # Biểu đồ top niche rules
    top_display = niche_rules.head(min(15, len(niche_rules))).copy()
    top_display['rule_label'] = top_display.apply(
        lambda x: f"{x['antecedents_str'][:30]}...→{x['consequents_str'][:20]}..." 
        if len(x['antecedents_str']) > 30 else f"{x['antecedents_str']}→{x['consequents_str']}",
        axis=1
    )
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # Biểu đồ Lift
    sns.barplot(
        data=top_display,
        y='rule_label',
        x='lift',
        palette='rocket',
        ax=axes[0]
    )
    axes[0].set_title('Top Luật Niche: Lift', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Lift', fontsize=12)
    axes[0].set_ylabel('Luật', fontsize=12)
    
    # Biểu đồ Confidence
    sns.barplot(
        data=top_display,
        y='rule_label',
        x='confidence',
        palette='mako',
        ax=axes[1]
    )
    axes[1].set_title('Top Luật Niche: Confidence', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Confidence', fontsize=12)
    axes[1].set_ylabel('')
    
    plt.tight_layout()
    
    # Export ảnh cho README
    import os
    os.makedirs('images', exist_ok=True)
    plt.savefig('images/herb_markers_bundle.png', dpi=300, bbox_inches='tight')
    print("✅ Đã lưu ảnh: images/herb_markers_bundle.png")
    
    plt.show()
    print("✓ Các luật niche có lift rất cao, cho thấy liên kết mạnh giữa sản phẩm")

---
## PHẦN 2: PHÂN TÍCH LÝ DO TẠI SAO CÁC LUẬT NÀY ĐÁNG CHÚ Ý

Các luật có **support thấp nhưng lift cao** rất quan trọng trong chiến lược bán hàng vì:

In [ ]:
print("PHÂN TÍCH GIÁ TRỊ CỦA LUẬT NICHE (Support thấp, Lift cao)")
print("="*80)

analysis_points = [
    {
        'title': '1. PHÂN KHÚC THỊ TRƯỜNG CHUYÊN BIỆT (Niche Market)',
        'description': [
            '• Support thấp: Chỉ một nhóm nhỏ khách hàng mua kết hợp này',
            '• Lift cao: Nhưng khi mua A, khả năng mua B tăng mạnh',
            '• Đây là dấu hiệu của phân khúc khách hàng đặc biệt với nhu cầu riêng',
            f'• Trong dữ liệu: {len(niche_rules)} luật thuộc loại này'
        ]
    },
    {
        'title': '2. GIÁ TRỊ KINH DOANH CAO',
        'description': [
            '• Khách hàng trong phân khúc này thường có giá trị cao',
            '• Họ tìm kiếm sản phẩm chuyên biệt, sẵn sàng trả giá cao hơn',
            '• Tỷ lệ mua thêm (cross-sell) cao do lift lớn',
            '• Ít cạnh tranh hơn so với thị trường đại trà'
        ]
    },
    {
        'title': '3. CƠ HỘI CHIẾN LƯỢC MARKETING',
        'description': [
            '• Bundle sản phẩm: Đóng gói A+B với giá ưu đãi',
            '• Gợi ý sản phẩm: Khi khách mua A, đề xuất B ngay lập tức',
            '• Quảng cáo nhắm mục tiêu: Tập trung vào phân khúc này',
            '• Tăng trải nghiệm khách hàng: Hiểu nhu cầu riêng của họ'
        ]
    },
    {
        'title': '4. KHÁC BIỆT VỚI LUẬT PHỔ BIẾN',
        'description': [
            '• Luật support cao: Phổ biến nhưng ít giá trị chiến lược',
            '• Luật niche: Hiếm nhưng có giá trị cao cho đúng khách hàng',
            '• ROI marketing cao hơn khi nhắm đúng phân khúc',
            '• Tạo lợi thế cạnh tranh bền vững'
        ]
    }
]

for point in analysis_points:
    print(f"\n{point['title']}")
    print("-" * 80)
    for desc in point['description']:
        print(desc)

print("\n" + "="*80)

In [ ]:
# Phân tích định lượng
if len(niche_rules) > 0:
    print("\nPHÂN TÍCH ĐỊNH LƯỢNG:")
    print("="*80)
    
    avg_lift_niche = niche_rules['lift'].mean()
    avg_lift_all = rules_df['lift'].mean()
    lift_improvement = ((avg_lift_niche - avg_lift_all) / avg_lift_all) * 100
    
    avg_conf_niche = niche_rules['confidence'].mean()
    avg_conf_all = rules_df['confidence'].mean()
    conf_improvement = ((avg_conf_niche - avg_conf_all) / avg_conf_all) * 100
    
    print(f"\n📊 So sánh Luật Niche vs Tất cả Luật:")
    print(f"   • Lift trung bình (Niche): {avg_lift_niche:.2f}")
    print(f"   • Lift trung bình (Tất cả): {avg_lift_all:.2f}")
    print(f"   • Cải thiện: {lift_improvement:.1f}% cao hơn")
    print(f"\n   • Confidence trung bình (Niche): {avg_conf_niche:.4f} ({avg_conf_niche*100:.2f}%)")
    print(f"   • Confidence trung bình (Tất cả): {avg_conf_all:.4f} ({avg_conf_all*100:.2f}%)")
    print(f"   • Cải thiện: {conf_improvement:.1f}% cao hơn")
    
    # Phân tích leverage
    avg_leverage_niche = niche_rules['leverage'].mean()
    print(f"\n   • Leverage trung bình (Niche): {avg_leverage_niche:.6f}")
    print(f"     → Mức độ tăng khả năng mua cùng nhau cao")
    
    # Tính tỷ lệ niche rules
    niche_percentage = (len(niche_rules) / len(rules_df)) * 100
    print(f"\n📈 Tỷ lệ luật niche: {niche_percentage:.2f}% ({len(niche_rules)}/{len(rules_df)} luật)")
    print(f"   → Đây là phân khúc 'kim cương' cần chú trọng đầu tư")

In [ ]:
# Phân tích các nhóm sản phẩm trong niche rules
if len(niche_rules) > 0:
    print("\n" + "="*80)
    print("PHÂN TÍCH CÁC NHÓM SẢN PHẨM TRONG LUẬT NICHE")
    print("="*80)
    
    # Trích xuất các sản phẩm từ antecedents và consequents
    all_products = []
    for idx, row in niche_rules.iterrows():
        # Antecedents
        if pd.notna(row['antecedents_str']):
            products = [p.strip() for p in str(row['antecedents_str']).split(',')]
            all_products.extend(products)
        # Consequents
        if pd.notna(row['consequents_str']):
            products = [p.strip() for p in str(row['consequents_str']).split(',')]
            all_products.extend(products)
    
    # Đếm tần suất
    from collections import Counter
    product_counts = Counter(all_products)
    
    print(f"\nTOP 15 SẢN PHẨM XUẤT HIỆN NHIỀU NHẤT TRONG LUẬT NICHE:")
    print("-" * 80)
    for product, count in product_counts.most_common(15):
        percentage = (count / len(niche_rules)) * 100
        print(f"  {product[:50]:50s} : {count:3d} lần ({percentage:5.1f}%)")
    
    # Phân tích category (nếu có pattern)
    print(f"\n💡 NHẬN XÉT:")
    print(f"   • Các sản phẩm này thường được mua kèm với nhau")
    print(f"   • Đây là cơ hội để tạo bundle hoặc promotion")
    print(f"   • Nên đặt các sản phẩm này gần nhau trong cửa hàng")

---
## PHẦN 3: ĐỀ XUẤT CHIẾN LƯỢC BÁN HÀNG CHUYÊN SÂU THEO PHÂN KHÚC

Dựa trên phân tích các luật niche, đưa ra các chiến lược kinh doanh cụ thể.

In [ ]:
print("CHIẾN LƯỢC BÁN HÀNG CHUYÊN SÂU THEO PHÂN KHÚC")
print("="*80)

strategies = [
    {
        'category': '🎯 CHIẾN LƯỢC 1: PHÂN KHÚC VÀ NHẮM MỤC TIÊU',
        'strategies': [
            {
                'name': 'A. Phân khúc khách hàng',
                'actions': [
                    'Xác định khách hàng mua các sản phẩm trong luật niche',
                    'Tạo profile chi tiết: Nhân khẩu học, hành vi mua sắm, giá trị đơn hàng',
                    'Gắn nhãn "Premium Niche Customer" trong CRM',
                    'Theo dõi lifetime value của phân khúc này'
                ]
            },
            {
                'name': 'B. Marketing nhắm mục tiêu',
                'actions': [
                    'Email marketing cá nhân hóa với sản phẩm liên quan',
                    'Retargeting ads cho khách đã xem sản phẩm A, đề xuất sản phẩm B',
                    'Chương trình loyalty đặc biệt cho phân khúc niche',
                    'Content marketing về use-case kết hợp sản phẩm'
                ]
            }
        ]
    },
    {
        'category': '🎁 CHIẾN LƯỢC 2: BUNDLE VÀ CROSS-SELLING',
        'strategies': [
            {
                'name': 'A. Sản phẩm Bundle',
                'actions': [
                    'Tạo combo sản phẩm dựa trên luật có lift cao nhất',
                    'Giảm giá bundle 10-15% so với mua lẻ',
                    'Thiết kế packaging đặc biệt cho bundle niche',
                    'Test A/B giá bundle để tối ưu doanh thu'
                ]
            },
            {
                'name': 'B. Recommendation Engine',
                'actions': [
                    'Khi khách thêm sản phẩm A vào giỏ, hiện "Thường được mua kèm"',
                    'Ưu tiên đề xuất sản phẩm có lift cao nhất',
                    'Hiển thị số % khách hàng mua kèm để tăng social proof',
                    'Offer "Mua kèm giảm 5%" cho sản phẩm được gợi ý'
                ]
            }
        ]
    },
    {
        'category': '🏪 CHIẾN LƯỢC 3: MERCHANDISING VÀ TRẢI NGHIỆM',
        'strategies': [
            {
                'name': 'A. Bố trí cửa hàng (Physical/Online)',
                'actions': [
                    'Đặt sản phẩm trong cùng luật gần nhau trên kệ',
                    'Tạo "Niche Corner" cho từng phân khúc chuyên biệt',
                    'Online: Tạo landing page "Collections" cho niche segments',
                    'Signage/Banner nhấn mạnh "Perfect Pair" hoặc "Complete Set"'
                ]
            },
            {
                'name': 'B. Trải nghiệm mua sắm',
                'actions': [
                    'Demo sử dụng kết hợp sản phẩm A+B',
                    'Video hướng dẫn "How to use together"',
                    'Testimonial từ khách hàng đã mua combo',
                    'Workshop/Event cho cộng đồng niche'
                ]
            }
        ]
    },
    {
        'category': '💰 CHIẾN LƯỢC 4: PRICING VÀ PROMOTION',
        'strategies': [
            {
                'name': 'A. Chiến lược giá',
                'actions': [
                    'Premium pricing cho bundle niche (khách sẵn sàng trả giá cao)',
                    'Dynamic pricing: Giảm giá sản phẩm B khi khách mua A',
                    'Tiered pricing: Bundle 2 sản phẩm, 3 sản phẩm với giá tối ưu',
                    'Subscription model cho khách mua định kỳ'
                ]
            },
            {
                'name': 'B. Chương trình khuyến mãi',
                'actions': [
                    '"Mua A tặng voucher 20% cho B" (sản phẩm trong luật niche)',
                    'Flash sale bundle vào thời điểm cao điểm',
                    'First-time buyer offer cho niche bundle',
                    'Referral program: Giới thiệu bạn mua combo được thưởng'
                ]
            }
        ]
    },
    {
        'category': '📊 CHIẾN LƯỢC 5: ĐO LƯỜNG VÀ TỐI ƯU',
        'strategies': [
            {
                'name': 'A. KPIs cần theo dõi',
                'actions': [
                    'Tỷ lệ chuyển đổi từ sản phẩm A sang A+B',
                    'Average Order Value (AOV) của phân khúc niche',
                    'Customer Lifetime Value (CLV) theo segment',
                    'Attachment rate (% khách mua thêm khi được gợi ý)'
                ]
            },
            {
                'name': 'B. Tối ưu liên tục',
                'actions': [
                    'A/B test các chiến lược bundle khác nhau',
                    'Re-run Apriori hàng tháng để cập nhật luật mới',
                    'Survey khách hàng niche về nhu cầu và mong đợi',
                    'Điều chỉnh chiến lược dựa trên feedback và data'
                ]
            }
        ]
    }
]

# In ra các chiến lược
for strategy in strategies:
    print(f"\n{strategy['category']}")
    print("=" * 80)
    for sub_strategy in strategy['strategies']:
        print(f"\n{sub_strategy['name']}")
        print("-" * 80)
        for i, action in enumerate(sub_strategy['actions'], 1):
            print(f"  {i}. {action}")

In [ ]:
# Tạo action plan cụ thể cho top niche rules
if len(niche_rules) > 0:
    print("\n" + "="*80)
    print("ACTION PLAN CỤ THỂ CHO TOP 5 LUẬT NICHE")
    print("="*80)
    
    top_5_niche = niche_rules.head(5)
    
    for i, (idx, row) in enumerate(top_5_niche.iterrows(), 1):
        print(f"\n{'─'*80}")
        print(f"LUẬT #{i}: {row['antecedents_str']} → {row['consequents_str']}")
        print(f"  Lift: {row['lift']:.2f} | Confidence: {row['confidence']:.2%} | Support: {row['support']:.4f}")
        print(f"{'─'*80}")
        
        print(f"\n📋 Action Items:")
        print(f"\n  ✅ NGAY LẬP TỨC (Tuần 1-2):")
        print(f"     1. Tạo bundle: '{row['antecedents_str']} + {row['consequents_str']}'")
        print(f"     2. Giảm giá bundle 12% để test thị trường")
        print(f"     3. Setup recommendation: Khi mua '{row['antecedents_str']}' → gợi ý '{row['consequents_str']}'")
        print(f"     4. Email đến khách đã mua '{row['antecedents_str']}' với offer mua thêm")
        
        print(f"\n  🎯 NGẮN HẠN (Tháng 1):")
        print(f"     1. Tạo landing page cho bundle này")
        print(f"     2. Facebook/Google Ads nhắm đến audience quan tâm sản phẩm này")
        print(f"     3. Đo lường conversion rate và AOV")
        print(f"     4. Đặt sản phẩm gần nhau trong cửa hàng (nếu có)")
        
        print(f"\n  📈 DÀI HẠN (Quý 1):")
        print(f"     1. Phát triển product line bổ sung cho phân khúc này")
        print(f"     2. Xây dựng loyalty program riêng")
        print(f"     3. Tổ chức workshop/event cho cộng đồng niche")
        print(f"     4. Tối ưu dựa trên dữ liệu thu thập được")
        
        expected_impact = row['confidence'] * 100
        print(f"\n  💡 Kỳ vọng: Tăng {expected_impact:.0f}% khả năng mua thêm '{row['consequents_str']}'")
        print(f"     khi khách hàng đã mua '{row['antecedents_str']}'")

In [ ]:
if PLOT_SEGMENT_ANALYSIS and len(niche_rules) > 0:
    # Phân tích phân khúc dựa trên support và lift
    
    # Tạo các phân khúc
    def categorize_rule(row):
        if row['support'] < LOW_SUPPORT_THRESHOLD and row['lift'] > HIGH_LIFT_THRESHOLD:
            return 'Niche Premium (Low Support, High Lift)'
        elif row['support'] >= LOW_SUPPORT_THRESHOLD and row['lift'] > HIGH_LIFT_THRESHOLD:
            return 'Popular Strong (High Support, High Lift)'
        elif row['support'] < LOW_SUPPORT_THRESHOLD and row['lift'] <= HIGH_LIFT_THRESHOLD:
            return 'Weak Niche (Low Support, Low Lift)'
        else:
            return 'Popular Weak (High Support, Low Lift)'
    
    rules_df['segment'] = rules_df.apply(categorize_rule, axis=1)
    
    # Đếm số lượng luật theo segment
    segment_counts = rules_df['segment'].value_counts()
    
    # Biểu đồ phân bố segments
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Phân Bố Số Lượng Luật Theo Phân Khúc', 'Giá Trị Trung Bình Theo Phân Khúc'),
        specs=[[{'type': 'pie'}, {'type': 'bar'}]]
    )
    
    # Pie chart
    fig.add_trace(
        go.Pie(
            labels=segment_counts.index,
            values=segment_counts.values,
            hole=0.3,
            marker=dict(colors=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'])
        ),
        row=1, col=1
    )
    
    # Bar chart - Average metrics by segment
    segment_metrics = rules_df.groupby('segment')[['lift', 'confidence']].mean()
    
    fig.add_trace(
        go.Bar(
            x=segment_metrics.index,
            y=segment_metrics['lift'],
            name='Avg Lift',
            marker_color='#FF6B6B'
        ),
        row=1, col=2
    )
    
    fig.update_layout(
        title_text='Phân Tích Phân Khúc Luật Kết Hợp',
        showlegend=True,
        height=500,
        width=1200
    )
    
    fig.show()
    
    print("\n📊 PHÂN TÍCH PHÂN KHÚC:")
    print("="*80)
    for segment in segment_counts.index:
        count = segment_counts[segment]
        percentage = (count / len(rules_df)) * 100
        avg_lift = rules_df[rules_df['segment'] == segment]['lift'].mean()
        avg_conf = rules_df[rules_df['segment'] == segment]['confidence'].mean()
        
        print(f"\n{segment}:")
        print(f"  • Số lượng: {count} luật ({percentage:.1f}%)")
        print(f"  • Lift TB: {avg_lift:.2f}")
        print(f"  • Confidence TB: {avg_conf:.2%}")
    
    print(f"\n💡 KHUYẾN NGHỊ ƯU TIÊN:")
    print(f"  1. Niche Premium: Đầu tư marketing chuyên sâu, premium pricing")
    print(f"  2. Popular Strong: Chiến lược mass market, volume-based")
    print(f"  3. Weak Niche: Xem xét loại bỏ hoặc test thêm")
    print(f"  4. Popular Weak: Maintain nhưng không ưu tiên đầu tư")

In [ ]:
# Export kết quả phân tích
if len(niche_rules) > 0:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Export niche rules
    niche_output_path = os.path.join(OUTPUT_DIR, 'niche_rules_analysis.csv')
    niche_rules.to_csv(niche_output_path, index=False)
    print(f"\n✅ Đã lưu phân tích luật niche vào: {niche_output_path}")
    
    # Export segments
    segment_output_path = os.path.join(OUTPUT_DIR, 'rules_by_segment.csv')
    rules_df.to_csv(segment_output_path, index=False)
    print(f"✅ Đã lưu phân tích phân khúc vào: {segment_output_path}")
    
    # Export action plan cho top rules
    top_action_plan = niche_rules.head(10)[[
        'antecedents_str', 'consequents_str', 'support', 'confidence', 'lift', 'leverage'
    ]].copy()
    top_action_plan['recommended_action'] = 'Create bundle + Recommendation + Email campaign'
    top_action_plan['priority'] = ['High' if i < 5 else 'Medium' for i in range(len(top_action_plan))]
    
    action_output_path = os.path.join(OUTPUT_DIR, 'niche_action_plan.csv')
    top_action_plan.to_csv(action_output_path, index=False)
    print(f"✅ Đã lưu action plan vào: {action_output_path}")
    
    print(f"\n{'='*80}")
    print("HOÀN THÀNH PHÂN TÍCH CHUYÊN SÂU LUẬT NICHE!")
    print(f"{'='*80}")

---
## KẾT LUẬN

### Tóm tắt phân tích:

1. **Xác định được các luật niche** (support thấp, lift cao) đại diện cho phân khúc thị trường chuyên biệt

2. **Giá trị kinh doanh**:
   - Lift cao → Khả năng cross-sell mạnh
   - Support thấp → Ít cạnh tranh, khách hàng premium
   - Cơ hội tạo lợi thế cạnh tranh bền vững

3. **Chiến lược đề xuất**:
   - **Phân khúc & Targeting**: Xác định và chăm sóc khách hàng niche
   - **Bundle & Cross-sell**: Tạo combo sản phẩm, recommendation engine
   - **Merchandising**: Bố trí sản phẩm, tạo trải nghiệm đặc biệt
   - **Pricing**: Premium pricing, dynamic pricing cho niche
   - **Đo lường**: KPIs và tối ưu liên tục

4. **Action Plan**: Roadmap cụ thể cho từng luật niche (Ngay lập tức → Ngắn hạn → Dài hạn)

### Bước tiếp theo:
- Triển khai pilot cho top 5 luật niche
- Đo lường ROI sau 1 tháng
- Scale up các chiến lược hiệu quả
- Cập nhật phân tích định kỳ hàng tháng